# Exploring Protspect residue representations

An editable version of the embedding atlas: compare individual channels and whole matrix representations using PCA, UMAP, clustering, and original-space neighbors.

**Start:** choose a Python kernel with `research/embedding_atlas/requirements.txt` plus the notebook requirements installed, then **Run All**. Edit the controls immediately above each analysis. Re-run only that stage and its plots when possible; downstream variables do not update automatically in Jupyter.

- Changing source families, orientation, sign invariance, or deduplication: rerun that section from preparation onward.
- Changing PCA controls: rerun its fit and plot cells.
- Changing UMAP controls: rerun its fit and plot cells. Fixed-seed results are cached by input distances **and** effective parameters; plotting changes need no new fit.
- Changing cluster cuts/linkage: rerun clustering, plots, and medoids; UMAP need not be refitted.
- Exports are **off by default** and never overwrite the reference atlas.

No peptide labels enter this analysis. Similar embeddings need not produce interchangeable spectral predictors. Vector profiles are centered over the 20 residue types and unit-normalized; whole geometry uses centered residue Gram matrices. Raw score units and constant offsets can still matter for actual spectral features.

In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts at the repository root or inside this notebook folder.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").exists() and (p / "protspect").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository or one of its subdirectories.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CACHE = ROOT / ".aaindex" / "notebook_cache"
for name, subdir in [("MPLCONFIGDIR", "matplotlib"), ("NUMBA_CACHE_DIR", "numba")]:
    (CACHE / subdir).mkdir(parents=True, exist_ok=True)
    os.environ.setdefault(name, str(CACHE / subdir))
os.environ.setdefault("NUMBA_NUM_THREADS", "1")

import hashlib
import json
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Image, display
from scipy.cluster.hierarchy import dendrogram, leaves_list
from threadpoolctl import threadpool_limits

from protspect.aaindex import (
    AAIndex, AAIndex2, AAIndex3,
    download_aaindex1, download_aaindex2, download_aaindex3,
)
from research.embedding_atlas.analyze import ALPHABET, COLORS, load_inputs
from research.embedding_atlas.explore import (
    cka_clustermap, aligned_matrix_features, cached_umap, cluster_profiles, medoids, neighbor_table,
    prepare_vectors, prepare_whole, project_pca, quality, scatter, umap_signature,
)

%matplotlib inline
_thread_limit = threadpool_limits(limits=1)
pio.renderers.default = "notebook"  # Offline JS, no CDN connection required.
UMAP_CACHE = globals().get("UMAP_CACHE", {})
def array_signature(array):
    return hashlib.sha256(np.ascontiguousarray(array).tobytes()).hexdigest()
pd.set_option("display.max_colwidth", 95)
print("Repository:", ROOT)
print("Kernel Python:", sys.executable)

## Data and display controls

These default paths use the ignored local `.aaindex/` cache. The snapshots are already present in this workspace. In another checkout, either copy your local files there or explicitly set `DOWNLOAD_MISSING=True`. Importing the package, loading existing files, and fitting models do not download data. The optional checksum check pins the original study snapshots.

In [ ]:
INPUT_PATHS = [ROOT / ".aaindex" / f"aaindex{i}" for i in (1, 2, 3)]
DOWNLOAD_MISSING = False
VERIFY_REFERENCE_HASHES = True
REFERENCE_HASHES = [
    "560f102dbee8ca9837186c4b2b93e253ea6efc58c88617234cef4e54bd61f676",
    "d3eb30c075452aa737b111a3d127358d0b669d0e847fca967b253a05c7796aa9",
    "8c9b9b0e7b7937b4674585b25f3f9370499f219b2583b1204fc9a0a5f06b1671",
]
SHOW_REFERENCE_FIGURES = True
TABLE_ROWS = 12
# BLOSUM100: red stars; Kidera: purple X markers; both outlined in black.
PLOT_STYLE = dict(marker_size=5, reference_size=11, opacity=0.55, height=620,
                  colors=dict(COLORS))  # Edit sizes, opacity, height, or family colors.
QUALITY_NEIGHBORS = 10
QUALITY_AUDIT_SIZE = 1000
USE_LAYOUT_CACHE = True
REFERENCE_DIR = ROOT / "validation" / "embedding_atlas"

In [ ]:
classes = [AAIndex, AAIndex2, AAIndex3]
downloaders = [download_aaindex1, download_aaindex2, download_aaindex3]
source_manifest = []
for i, (path, catalogue, download) in enumerate(zip(INPUT_PATHS, classes, downloaders)):
    expected = REFERENCE_HASHES[i] if VERIFY_REFERENCE_HASHES else None
    if not path.exists():
        if not DOWNLOAD_MISSING:
            raise FileNotFoundError(f"{path}: provide a local file or set DOWNLOAD_MISSING=True.")
        download(path, expected_sha256=expected)
    database = catalogue.from_file(path, expected_sha256=expected)
    source_manifest.append(dict(section=f"AAindex{i + 1}", path=str(path),
                                records=len(database), sha256=database.sha256))
display(pd.DataFrame(source_manifest))
databases, source_catalogue, source_profiles, source_whole, exclusions = load_inputs(INPUT_PATHS)
display(source_catalogue.groupby("family").size().rename("valid_channel_instances"))
display(exclusions.groupby(["kind", "reason"]).size().rename("excluded"))
print("All complete individual profiles are retained, even when another cell of their parent matrix is missing.")

In [ ]:
if SHOW_REFERENCE_FIGURES:
    display(Image(filename=str(REFERENCE_DIR / "whole_overview.png")))
    display(Image(filename=str(REFERENCE_DIR / "vectors_overview.png")))
    display(pd.read_csv(REFERENCE_DIR / "cluster_thresholds.csv"))

## Individual channels: choose the comparison

`VECTOR_SIGN_INVARIANT=True` compares projectors `v vᵀ`, so similarity equals **r²** and sign-reversed profiles coincide. `False` uses signed unit profiles, with similarity **r**. Both start from centered 20-residue profiles. Changing this choice changes the clustering metric and layouts.

Exact/affine-shape aliases are retained in the catalogue even when deduplicated for plotting. Shared point colors prioritize Kidera, BLOSUM100, AAindex1, AAindex2, AAindex3; a color is not an exclusive source label. BLOSUM100 uses red stars with black outlines; Kidera uses purple X markers with black outlines in the computed plots.


### Kidera factor descriptions

The atlas uses these property descriptions in hover text and tables; stable
channel IDs remain `KF1`–`KF10`. These are conventional labels, not independent
measurements of each named property. In particular, KF5–KF10 combine several
properties and are named for their most heavily weighted component. See the
[Peptides reference documentation](https://search.r-project.org/CRAN/refmans/Peptides/html/kideraFactors.html).

| Factor | Property description |
| --- | --- |
| KF1 | Helix/bend preference |
| KF2 | Side-chain size |
| KF3 | Extended structure preference |
| KF4 | Hydrophobicity |
| KF5 | Double-bend preference |
| KF6 | Partial specific volume |
| KF7 | Flat extended preference |
| KF8 | Occurrence in alpha region |
| KF9 | Carboxyl-group pK (pK-C) |
| KF10 | Surrounding hydrophobicity |



In [ ]:
VECTOR_FAMILIES = ["AAindex1", "AAindex2", "AAindex3", "BLOSUM100", "Kidera"]
VECTOR_AXIS = "both"  # "both", "rows", "columns"; scalar/reference channels remain included.
VECTOR_SIGN_INVARIANT = True
VECTOR_DEDUPLICATE = True
VECTOR_DEDUP_DECIMALS = 10

vector_aliases, vector_points, vector_profiles, vector_features, vector_similarity, vector_distance = prepare_vectors(
    source_catalogue, source_profiles, VECTOR_FAMILIES,
    sign_invariant=VECTOR_SIGN_INVARIANT, deduplicate=VECTOR_DEDUPLICATE,
    decimals=VECTOR_DEDUP_DECIMALS, axis=VECTOR_AXIS,
)
print(f"{len(vector_aliases):,} channel instances → {len(vector_points):,} plotted shapes")
print("Similarity:", "squared Pearson correlation" if VECTOR_SIGN_INVARIANT else "signed Pearson correlation")
display(vector_points[["id", "family", "alias_count", "description"]].head(TABLE_ROWS))

### Vector PCA

Increase components to inspect the cumulative variance curve. Change `VECTOR_PCA_AXES` to view another pair, using zero-based component indices. A two-dimensional projection can miss substantial variation; use original-space distances for neighbors and clustering.

In [ ]:
VECTOR_PCA_COMPONENTS = 10
VECTOR_PCA_AXES = (0, 1)
vector_pca_scores, vector_pca = project_pca(vector_features, VECTOR_PCA_COMPONENTS)
vector_pca_input = array_signature(vector_features)
variance = vector_pca.explained_variance_ratio_
display(pd.DataFrame({"component": np.arange(1, len(variance) + 1),
                      "explained_fraction": variance, "cumulative_fraction": variance.cumsum()}))
x_axis, y_axis = VECTOR_PCA_AXES
if max(x_axis, y_axis) >= vector_pca_scores.shape[1] or min(x_axis, y_axis) < 0:
    raise ValueError("Choose PCA axes within the fitted component range.")
fig = scatter(vector_pca_scores[:, [x_axis, y_axis]], vector_points,
              title="Individual-channel PCA",
              x_label=f"PC{x_axis + 1} ({variance[x_axis]:.1%})",
              y_label=f"PC{y_axis + 1} ({variance[y_axis]:.1%})", **PLOT_STYLE)
fig.show()

### Vector UMAP

Useful controls include `n_neighbors`, `min_dist`, `spread`, `n_epochs`, and `random_state`. Larger neighborhoods emphasize broader structure. Set `random_state=None` for a new stochastic fit; that disables layout reuse. All fits use the original selected distances, not PCA coordinates. Neighbors are capped at the number of available points minus one and the effective parameters are printed.

In [ ]:
VECTOR_UMAP = dict(n_neighbors=15, min_dist=0.15, spread=1.0,
                   n_epochs=300, random_state=0, init="spectral")
vector_umap, vector_umap_effective, vector_umap_key, cache_hit = cached_umap(
    vector_distance, VECTOR_UMAP, UMAP_CACHE, use_cache=USE_LAYOUT_CACHE)
print("Cached layout reused:", cache_hit)
print("Effective parameters:", vector_umap_effective)

In [ ]:
scatter(vector_umap, vector_points, title="Individual channels: UMAP",
        x_label="UMAP 1", y_label="UMAP 2", **PLOT_STYLE).show()
display(pd.DataFrame([
    dict(layout="PCA axes 1–2", **quality(vector_distance, vector_pca_scores[:, :2],
         neighbors=QUALITY_NEIGHBORS, audit_size=QUALITY_AUDIT_SIZE)),
    dict(layout="UMAP", **quality(vector_distance, vector_umap,
         neighbors=QUALITY_NEIGHBORS, audit_size=QUALITY_AUDIT_SIZE)),
]))

### Vector clustering and representatives

With **complete linkage**, every pair inside a cluster satisfies the chosen correlation cut (absolute correlation in sign-invariant mode). Average/single/Ward cuts have no such pairwise guarantee. Ward is available because these are Euclidean feature-space distances, but its merge heights are not pairwise correlations.

The original AAindex1-only curve is shown above. Filtering to `VECTOR_FAMILIES=["AAindex1"]` and rerunning this section reproduces that separate candidate pool.

In [ ]:
VECTOR_LINKAGE = "complete"  # "complete", "average", "single", "ward"
VECTOR_CLUSTER_CUTS = [0.80, 0.90, 0.95, 0.98, 0.99]
VECTOR_ACTIVE_CUT = 0.95  # Must be one of the cuts above.
VECTOR_DENDROGRAM_LEAVES = 30

vector_tree, vector_cluster_curve, vector_assignments = cluster_profiles(
    vector_distance, VECTOR_CLUSTER_CUTS, sign_invariant=VECTOR_SIGN_INVARIANT,
    method=VECTOR_LINKAGE)
if VECTOR_ACTIVE_CUT not in vector_assignments:
    raise ValueError("Add VECTOR_ACTIVE_CUT to VECTOR_CLUSTER_CUTS.")
vector_cluster_input = array_signature(vector_distance)
vector_clusters = vector_assignments[VECTOR_ACTIVE_CUT]
vector_representatives = medoids(vector_points, vector_distance, vector_clusters)
display(vector_cluster_curve)
display(vector_representatives[["id", "family", "cluster", "unique_members", "description"]].head(TABLE_ROWS))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
axes[0].plot(vector_cluster_curve.cut_similarity, vector_cluster_curve.clusters, "o-")
axes[0].set(xlabel="Similarity cut (r threshold; absolute when sign-invariant)", ylabel="Clusters",
            title=f"{VECTOR_LINKAGE} linkage")
dendrogram(vector_tree, truncate_mode="lastp", p=VECTOR_DENDROGRAM_LEAVES,
           show_leaf_counts=True, leaf_rotation=90, ax=axes[1])
axes[1].set(title="Truncated vector dendrogram (contracted leaves show counts)", ylabel="Original-space distance")
plt.show()

### Inspect a channel and its aliases/neighbors

Try `Kidera:KF4`, `KIDA850101`, `BLOSUM100:A`, or a matrix profile such as `HENS920103:rows:A`. Exact aliases are shown separately because they occupy the same plotted point. Neighbors are ranked in the original chosen metric, never from UMAP coordinates.

In [ ]:
VECTOR_QUERY = "Kidera:KF4"
VECTOR_COMPARE = "KIDA850101"
PROFILE_PLOT_SCALE = "centered_unit"  # "centered_unit" or "raw"

match = vector_aliases[vector_aliases.id == VECTOR_QUERY]
if match.empty:
    print("Query not in selected families; using the first selected source channel.")
    match = vector_aliases.iloc[:1]
uid = int(match.iloc[0].unique_id)
representative_id = vector_points.iloc[uid].id
display(vector_aliases[vector_aliases.unique_id == uid][["id", "family", "description"]])
display(neighbor_table(representative_id, vector_points, vector_similarity, count=TABLE_ROWS)[
    ["similarity", "id", "family", "alias_count", "description"]])
fig, ax = plt.subplots(figsize=(13, 4))
for identifier in [match.iloc[0].id, VECTOR_COMPARE]:
    hit = np.flatnonzero(source_catalogue.id.to_numpy() == identifier)
    if not len(hit):
        print("Profile not found:", identifier)
        continue
    i = int(hit[0])
    values = source_profiles[i].copy()
    if PROFILE_PLOT_SCALE == "raw":
        values = values * source_catalogue.iloc[i].population_std * np.sqrt(20) + source_catalogue.iloc[i]["mean"]
    elif PROFILE_PLOT_SCALE != "centered_unit":
        raise ValueError("Choose centered_unit or raw profile scale.")
    ax.plot(list(ALPHABET), values, "o-", label=identifier)
ax.set(xlabel="Residue", ylabel=PROFILE_PLOT_SCALE, title="Selected residue-property profiles")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## Whole representations

CKA compares the residue geometry of a representation, including different channel counts. `WHOLE_EQUALIZE_CHANNELS=True` removes each channel's individual norm before forming the centered residue Gram; `False` keeps native relative channel weighting. Both remain invariant to a global scale. No whitening is applied.

CKA similarity does not imply identical per-channel Fourier amplitudes: rotating a channel basis can preserve the Gram while changing its magnitude features. The scalar/vector analysis remains complementary.

In [ ]:
WHOLE_FAMILIES = ["AAindex1", "AAindex2", "AAindex3", "BLOSUM100", "Kidera"]
WHOLE_AXIS = "both"  # Asymmetric orientations only; symmetric profiles are represented once.
WHOLE_EQUALIZE_CHANNELS = True
whole_points, whole_features, whole_similarity, whole_distance, chosen_whole = prepare_whole(
    source_whole, WHOLE_FAMILIES, equal_channels=WHOLE_EQUALIZE_CHANNELS, axis=WHOLE_AXIS)
print("Whole configurations:", len(whole_points))
display(whole_points[["id", "family", "orientation", "channels", "rank", "effective_rank"]].head(TABLE_ROWS))

In [ ]:
WHOLE_PCA_COMPONENTS = 10
WHOLE_PCA_AXES = (0, 1)
whole_pca_scores, whole_pca = project_pca(whole_features, WHOLE_PCA_COMPONENTS)
whole_pca_input = array_signature(whole_features)
whole_variance = whole_pca.explained_variance_ratio_
x_axis, y_axis = WHOLE_PCA_AXES
if max(x_axis, y_axis) >= whole_pca_scores.shape[1] or min(x_axis, y_axis) < 0:
    raise ValueError("Choose whole PCA axes within the fitted component range.")
scatter(whole_pca_scores[:, [x_axis, y_axis]], whole_points, title="Whole residue-geometry PCA",
        x_label=f"PC{x_axis + 1} ({whole_variance[x_axis]:.1%})",
        y_label=f"PC{y_axis + 1} ({whole_variance[y_axis]:.1%})", **PLOT_STYLE).show()
display(pd.DataFrame({"component": np.arange(1, len(whole_variance) + 1),
                      "explained_fraction": whole_variance,
                      "cumulative_fraction": whole_variance.cumsum()}))

In [ ]:
WHOLE_UMAP = dict(n_neighbors=15, min_dist=0.15, spread=1.0,
                  n_epochs=500, random_state=0, init="spectral")
whole_umap, whole_umap_effective, whole_umap_key, cache_hit = cached_umap(
    whole_distance, WHOLE_UMAP, UMAP_CACHE, use_cache=USE_LAYOUT_CACHE)
print("Cached layout reused:", cache_hit)
print("Effective parameters:", whole_umap_effective)

In [ ]:
scatter(whole_umap, whole_points, title="Whole residue geometry: UMAP",
        x_label="UMAP 1", y_label="UMAP 2", **PLOT_STYLE).show()
display(pd.DataFrame([dict(layout="Whole UMAP", **quality(
    whole_distance, whole_umap, neighbors=QUALITY_NEIGHBORS, audit_size=QUALITY_AUDIT_SIZE))]))

### Whole-representation clusters, heatmap, and nearest neighbors

The CKA cut below is different from a vector correlation cut. Only complete linkage guarantees the stated minimum pairwise CKA. Change heatmap range/palette to explore contrast; that does not change the similarity values.

Row and column dendrograms use the same original-space CKA linkage tree and leaf order. Branch heights show linkage distance, `sqrt(2 * (1 - CKA))`, rather than PCA/UMAP distances. Set `HEATMAP_SHOW_LABELS=True` to display IDs, or hover to inspect them; adjust `DENDROGRAM_FRACTION` to resize the trees.

In [ ]:
WHOLE_LINKAGE = "complete"
WHOLE_CLUSTER_CUTS = [0.80, 0.90, 0.95, 0.98, 0.99]
WHOLE_ACTIVE_CUT = 0.95
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_RANGE = (0.0, 1.0)
HEATMAP_HEIGHT = 900
HEATMAP_SHOW_LABELS = False
DENDROGRAM_FRACTION = 0.20
WHOLE_QUERY = "BLOSUM100"

whole_tree, whole_cluster_curve, whole_assignments = cluster_profiles(
    whole_distance, WHOLE_CLUSTER_CUTS, sign_invariant=False, method=WHOLE_LINKAGE)
if WHOLE_ACTIVE_CUT not in whole_assignments:
    raise ValueError("Add WHOLE_ACTIVE_CUT to WHOLE_CLUSTER_CUTS.")
whole_cluster_input = array_signature(whole_distance)
whole_clusters = whole_assignments[WHOLE_ACTIVE_CUT]
whole_representatives = medoids(whole_points, whole_distance, whole_clusters)
display(whole_cluster_curve)
if WHOLE_QUERY not in set(whole_points.id):
    print("Query not selected; using the first whole representation.")
    WHOLE_QUERY = whole_points.iloc[0].id
display(neighbor_table(WHOLE_QUERY, whole_points, whole_similarity, count=TABLE_ROWS)[
    ["similarity", "id", "family", "effective_rank", "description"]])
fig = cka_clustermap(whole_similarity, whole_points, whole_tree,
    colorscale=HEATMAP_COLORSCALE, color_range=HEATMAP_RANGE,
    height=HEATMAP_HEIGHT, show_labels=HEATMAP_SHOW_LABELS,
    dendrogram_fraction=DENDROGRAM_FRACTION)
fig.show()

### Inspect the selected representation's channel matrix

Rows below are sequence residues; columns are the representation's channels, matching the package's embedding convention. The raw profile matrix for an AAindex `:rows` configuration is the transpose of its native score matrix. This plot does not change the comparison settings. Limiting the display width only hides columns in the figure, not in the analysis.

In [ ]:
MAX_INSPECT_CHANNELS = 30
INSPECT_NORMALIZE_CHANNELS = False
selected = next(item for item in chosen_whole if item["id"] == WHOLE_QUERY)
inspection = selected["embedding"].copy()
if INSPECT_NORMALIZE_CHANNELS:
    inspection -= inspection.mean(axis=0)
    norms = np.linalg.norm(inspection, axis=0)
    inspection[:, norms > 1e-12] /= norms[norms > 1e-12]
if selected["id"] == "Kidera":
    channel_names = [f"KF{i}" for i in range(1, 11)]
elif selected["id"] == "AAindex1_complete_set":
    channel_names = [e.accession for e in databases[0].values() if e.complete]
else:
    channel_names = list(ALPHABET)
shown = min(MAX_INSPECT_CHANNELS, inspection.shape[1])
fig = go.Figure(go.Heatmap(z=inspection[:, :shown], y=list(ALPHABET), x=channel_names[:shown],
                          colorscale="RdBu_r", zmid=0, colorbar_title="Score"))
fig.update_layout(title=f"{WHOLE_QUERY}: {shown}/{inspection.shape[1]} channels displayed", height=550,
                  xaxis_title="Reference channel", yaxis_title="Sequence residue")
fig.show()

### Signed aligned-matrix PCA

This is complementary to CKA: only square canonical matrices with matching reference-residue axes are compared. Kidera and the full scalar-index family have different channel axes and are therefore excluded. Signs and channel labels are preserved; global score mean/scale are removed. AAindex BLOSUM62 uses 1/3-bit units, unlike NCBI's standard 1/2-bit file; the original source values are retained.

In [ ]:
SCORE_PCA_COMPONENTS = 10
SCORE_PCA_AXES = (0, 1)
aligned_points, aligned_features = aligned_matrix_features(chosen_whole)
aligned_scores, aligned_pca = project_pca(aligned_features, SCORE_PCA_COMPONENTS)
x_axis, y_axis = SCORE_PCA_AXES
if max(x_axis, y_axis) >= aligned_scores.shape[1] or min(x_axis, y_axis) < 0:
    raise ValueError("Choose score PCA axes within the fitted component range.")
scatter(aligned_scores[:, [x_axis, y_axis]], aligned_points, title="Signed aligned matrix-score PCA",
        x_label=f"PC{x_axis + 1} ({aligned_pca.explained_variance_ratio_[x_axis]:.1%})",
        y_label=f"PC{y_axis + 1} ({aligned_pca.explained_variance_ratio_[y_axis]:.1%})", **PLOT_STYLE).show()

## Export an experiment without overwriting the reference

Set `EXPORT_RUN=True` to save selected source aliases, coordinates, cluster representatives, whole similarities, and all controls in a fresh timestamped directory under the ignored `.aaindex/notebook_runs/` cache. Exports capture the current variables: rerun affected stages first after changing controls. No source databases are exported. Interactive figures can also be saved manually with `fig.write_html(...)` or through their toolbars.

In [ ]:
EXPORT_RUN = False
if EXPORT_RUN:
    checks = {
        "vector PCA": vector_pca_input == array_signature(vector_features),
        "vector clusters": vector_cluster_input == array_signature(vector_distance),
        "vector UMAP": vector_umap_key == umap_signature(vector_distance, VECTOR_UMAP)[2],
        "whole PCA": whole_pca_input == array_signature(whole_features),
        "whole clusters": whole_cluster_input == array_signature(whole_distance),
        "whole UMAP": whole_umap_key == umap_signature(whole_distance, WHOLE_UMAP)[2],
    }
    stale = [stage for stage, valid in checks.items() if not valid]
    if stale:
        raise RuntimeError("Rerun stale stages before exporting: " + ", ".join(stale))
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    run_dir = ROOT / ".aaindex" / "notebook_runs" / stamp
    run_dir.mkdir(parents=True, exist_ok=False)
    vector_aliases.assign(cluster=vector_clusters[vector_aliases.unique_id]).to_csv(
        run_dir / "vector_aliases.csv", index=False)
    vector_points.assign(PC1=vector_pca_scores[:, 0], PC2=vector_pca_scores[:, 1],
                         UMAP1=vector_umap[:, 0], UMAP2=vector_umap[:, 1],
                         cluster=vector_clusters).to_csv(run_dir / "vector_layout.csv", index=False)
    whole_points.assign(PC1=whole_pca_scores[:, 0], PC2=whole_pca_scores[:, 1],
                        UMAP1=whole_umap[:, 0], UMAP2=whole_umap[:, 1],
                        cluster=whole_clusters).to_csv(run_dir / "whole_layout.csv", index=False)
    vector_representatives.to_csv(run_dir / "vector_medoids.csv", index=False)
    whole_representatives.to_csv(run_dir / "whole_medoids.csv", index=False)
    pd.DataFrame(whole_similarity, index=whole_points.id, columns=whole_points.id).to_csv(
        run_dir / "whole_similarity.csv")
    controls = dict(sources=source_manifest, vector_families=VECTOR_FAMILIES, vector_axis=VECTOR_AXIS,
        sign_invariant=VECTOR_SIGN_INVARIANT, deduplicate=VECTOR_DEDUPLICATE,
        dedup_decimals=VECTOR_DEDUP_DECIMALS, vector_linkage=VECTOR_LINKAGE,
        vector_cluster_cuts=VECTOR_CLUSTER_CUTS, vector_active_cut=VECTOR_ACTIVE_CUT,
        vector_pca_components=VECTOR_PCA_COMPONENTS, vector_pca_axes=VECTOR_PCA_AXES,
        vector_umap=vector_umap_effective, vector_umap_signature=vector_umap_key,
        whole_families=WHOLE_FAMILIES, whole_axis=WHOLE_AXIS,
        whole_equalize_channels=WHOLE_EQUALIZE_CHANNELS, whole_linkage=WHOLE_LINKAGE,
        whole_cluster_cuts=WHOLE_CLUSTER_CUTS, whole_active_cut=WHOLE_ACTIVE_CUT,
        whole_pca_components=WHOLE_PCA_COMPONENTS, whole_pca_axes=WHOLE_PCA_AXES,
        whole_umap=whole_umap_effective, whole_umap_signature=whole_umap_key,
        score_pca_components=SCORE_PCA_COMPONENTS, score_pca_axes=SCORE_PCA_AXES,
        plot_style=PLOT_STYLE, quality_neighbors=QUALITY_NEIGHBORS,
        quality_audit_size=QUALITY_AUDIT_SIZE)
    (run_dir / "controls.json").write_text(json.dumps(controls, indent=2), encoding="utf-8")
    print("Saved experiment:", run_dir)
else:
    print("Export is off. Reference results have not been overwritten.")

## Interpretation and next steps

- At the reference settings, strict AAindex1 clustering at |r| ≥ 0.95 retains 436 of 553 complete properties; conservative redundancy removal does not create a tiny candidate pool.
- Two-dimensional layouts show only part of the geometry. Inspect neighbor recall, trustworthiness, aliases, and original-space similarities before drawing conclusions from apparent UMAP islands.
- Geometry similarity is an unsupervised candidate-reduction heuristic. Sequence composition, padding, spectral normalization, phase/cross spectra, and downstream tasks can change practical equivalence.
- Use exported representatives as an initial MISTIC search pool; keep within-cluster alternatives available for refinement.

See the [full methods](README.md), [reference results](../../validation/embedding_atlas/RESULTS.md), and [AAindex attribution](../../docs/AAINDEX.md#database-attribution).